# Loading and Evaluating Foundry LLM and VLM Models

This tutorial shows how to load pretrained LLM and VLM models from HuggingFace Hub and run quick inference checks. By the end you will be able to:
- Load a Foundry LLM and generate text completions
- Load a Foundry VLM and generate image captions
- Understand how these models connect to the full VLA training pipeline

## Prerequisites

- NVIDIA GPU
- The **`Python (vla_foundry)`** Jupyter kernel — install it once from the repo root:
  ```bash
  bash tutorials/install_kernel.sh
  ```
- Restart your notebook kernel after installing and select it for execution.


## Overview

| Step | What | Model |
|------|------|-------|
| 0 | Configuration | — |
| 1 | Load and evaluate the LLM | [TRI-ML/Foundry-LLM-1B-1T](https://huggingface.co/TRI-ML/Foundry-LLM-1B-1T) |
| 2 | Load and evaluate the VLM | [TRI-ML/Foundry-VLM-1.1B-200M](https://huggingface.co/TRI-ML/Foundry-VLM-1.1B-200M) |

---

## Step 0: Configuration

In [ ]:
import os
import subprocess

# Change to the repo root.
repo_root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
os.chdir(repo_root)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ---- Models ----
# HuggingFace model IDs. Models are downloaded and cached automatically on first use.
# Browse available models at: https://huggingface.co/TRI-ML
LLM_MODEL_ID = "TRI-ML/Foundry-LLM-1B-1T"
VLM_MODEL_ID = "TRI-ML/Foundry-VLM-1.1B-200M"

DEVICE = "cuda:0"

---

## Step 1: Load and Evaluate the LLM

We load the model using `BaseModel.from_pretrained`, which handles HuggingFace download, config parsing, and checkpoint loading in one call. The tokenizer and precision settings are read from the model's `config.yaml`.

In [ ]:
import torch
from transformers import AutoTokenizer

from vla_foundry.models.base_model import BaseModel

# Load model (downloads from HF Hub on first run, cached afterwards).
llm = BaseModel.from_pretrained(LLM_MODEL_ID, device=DEVICE)
print(f"LLM loaded: {sum(p.numel() for p in llm.parameters()) / 1e9:.2f}B parameters")

In [ ]:
# Load the tokenizer used during training.
from vla_foundry.hf_hub import resolve_hf_path
from vla_foundry.params.train_experiment_params import TrainExperimentParams, load_params_from_yaml

config_path = resolve_hf_path(f"hf://{LLM_MODEL_ID}/config.yaml")
train_params = load_params_from_yaml(TrainExperimentParams, config_path)

tokenizer = AutoTokenizer.from_pretrained(train_params.data.tokenizer)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
print(f"Tokenizer: {train_params.data.tokenizer}")

In [ ]:
# Generate text completions (temperature=0 for deterministic output).
from IPython.display import Markdown, display

prompts = [
    "The key to training generalist robot policies is ",
    "To find the avocado in the image, the robot should ",
]

precision = train_params.hparams.precision
llm_dtype = torch.bfloat16 if "bf16" in precision else torch.float16 if "fp16" in precision else torch.float32

torch.manual_seed(42)
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    prompt_len = inputs["input_ids"].shape[-1]

    with torch.autocast(device_type="cuda", dtype=llm_dtype), torch.no_grad():
        output = llm.generate(**inputs, max_new_tokens=80, temperature=0.0)

    completion = tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True)
    display(Markdown(f"**Prompt:** {prompt}\n\n**Completion:** {completion.strip()}\n\n---"))

In [ ]:
# Free GPU memory before loading the VLM.
del llm
torch.cuda.empty_cache()

---

## Step 2: Load and Evaluate the VLM

The VLM pairs the LLM backbone with a ViT image encoder and can generate text descriptions of images. The processor and its kwargs (image size, etc.) are read from the model's `config.yaml` to ensure the input matches what the model was trained on.

We center-crop images to square before passing them to the processor, since the processor resizes to a fixed resolution and non-square images would be distorted.

In [ ]:
from vla_foundry.data.processor import apply_chat_template, get_processor
from vla_foundry.models.base_model import BaseModel

# Load model.
vlm = BaseModel.from_pretrained(VLM_MODEL_ID, device=DEVICE)
print(f"VLM loaded: {sum(p.numel() for p in vlm.parameters()) / 1e9:.2f}B parameters")

In [ ]:
# Load the processor (tokenizer + image preprocessing) used during training.
config_path = resolve_hf_path(f"hf://{VLM_MODEL_ID}/config.yaml")
vlm_params = load_params_from_yaml(TrainExperimentParams, config_path)

processor = get_processor(vlm_params.data)
proc_kwargs = vlm_params.data.processor_kwargs
eos_id = processor.tokenizer.eos_token_id
prompt = apply_chat_template(processor, 1, "")  # 1 image, empty text prompt

precision = vlm_params.hparams.precision
dtype = torch.bfloat16 if "bf16" in precision else torch.float16 if "fp16" in precision else torch.float32
vlm = vlm.to(dtype)

print(f"Processor: {vlm_params.data.processor}")
print(f"Processor kwargs: {proc_kwargs}")
print(f"Precision: {dtype}")

In [ ]:
# Test on a mix of natural and robotics images.
from IPython.display import display

from vla_foundry.inference.inference_vlm import load_image

test_images = [
    ("cat", "tutorials/assets/cat.png"),
    ("dog", "tutorials/assets/dog.png"),
    ("robot tools", "tutorials/assets/robot_tools.png"),
    ("foundry_logo", "tutorials/assets/colored_logo.png"),
    ("robot scene", "tutorials/assets/stack_plates_top_right.png"),
    ("robot simulation", "tutorials/assets/BimanualPutSpatulaOnPlateFromUtensilCrock.png"),
]

print("=== VLM Captions ===\n")
for name, path in test_images:
    image = load_image(path, center_crop_square=True)

    # Show the image.
    print(f"[{name}]")
    display(image.resize((384, 384)))

    inputs = processor(image, prompt, return_tensors="pt", **proc_kwargs)
    if inputs["pixel_values"].dim() == 5:
        inputs["pixel_values"] = inputs["pixel_values"].squeeze(1)
    inputs.pop("pixel_attention_mask", None)

    # Strip trailing <end_of_utterance> + newline to match training format.
    if eos_id is not None and inputs["input_ids"].shape[-1] >= 2 and inputs["input_ids"][0, -2].item() == eos_id:
        inputs["input_ids"] = inputs["input_ids"][:, :-2]
        inputs["attention_mask"] = inputs["attention_mask"][:, :-2]

    inputs = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
    inputs["pixel_values"] = inputs["pixel_values"].to(dtype)

    with torch.no_grad():
        out = vlm.generate(**inputs, max_new_tokens=100, temperature=0.0, eos_token_id=eos_id)

    prompt_len = inputs["input_ids"].shape[-1]
    decoded = processor.decode(out[0][prompt_len:], skip_special_tokens=True)
    print(f"Caption: {decoded}\n")

---

## Next Steps

These models are the first two stages of the Foundry training pipeline:

```
Stage 1: LLM (text)  →  Stage 2: VLM (text + vision)  →  Stage 3: VLA (text + vision + actions)
```

- **Train your own models** — see [tutorials/training_llm_vlm_vla.ipynb](training_llm_vlm_vla.ipynb) for the full 3-stage pipeline.
- **Evaluate a VLA in simulation** — see [tutorials/sim_evaluation_tutorial.ipynb](sim_evaluation_tutorial.ipynb) to run simulation evaluation with Docker.
- **Browse models** — available at [huggingface.co/TRI-ML](https://huggingface.co/TRI-ML).